# How does Qwen Performance on the SAT?

## Importing the Model

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2-1.5B-Instruct" # This is the model we are using
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
model.generation_config.pad_token_id = tokenizer.pad_token_id

### Checkpoint #1
Let's take a step back and think critically about the model we chose to upload. Use these resources to help you answer the following question:

Article on Qwen 2-1.5
https://dataloop.ai/library/model/qwen_qwen2-15b/#:~:text=Performance%20The%20Qwen2%2D1.5B%20model%20is%20incredibly%20fast%2C,text%20classification%2C%20sentiment%20analysis%2C%20and%20language%20translation.

Article on Qwen 2-7
https://clarifai.com/qwen/qwenLM/models/qwen2-7b-chat

General Overview on Qwen Models
https://medium.com/@bgipradeep123/comparing-qwen-series-models-a-deep-dive-into-their-reasoning-capabilities-d8ee1bb0321e

In [4]:
# RUN THIS CELL TO ANSWER CHECKPOINT #1
user_response = input("Why do you think we imported Qwen2-1.5B compared to other Qwen models?\n")
file_path = 'answers.txt'
with open(file_path, 'w') as file:
    file.write(user_response)
    file.write('\n') 

Why do you think we imported Qwen2-1.5B compared to other Qwen models?
 storage space


In [ ]:
def ask_sat_question(question, options):
    # Construct the full prompt including the question, options, and instructions for the model
    prompt = f"""
    Please reason step by step, and put your final answer within \\boxed{{}}.
    Here is an SAT-style multiple-choice question:
    
    Question: {question}
    Options:
    {options}

    Please provide your detailed reasoning and select the single best answer.
    """
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    
    # Format the prompt into the required chat template for Qwen
    messages = [
        {"role": "system", "content": "You are a helpful assistant that excels at academic questions."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Generate the response
    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens = 512, 
        do_sample = True,
        temperature = 0.6,
        top_p = 0.95,
        pad_token_id=tokenizer.eos_token_id # Prevents generation issues if max length is hit
    )
    generated_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return response

This is an example of how we would ask Qwen to answer a sample question for us. Run it to see how Qwen responds!

In [8]:
question_text = "If 3x + 5 = 14, what is the value of x?"
options_text = "A) 2\nB) 3\nC) 4\nD) 5"

response = ask_sat_question(question_text, options_text)
print(response)

To solve this problem, we can start by isolating the variable \(x\) on one side of the equation. We can do this by subtracting 5 from both sides of the equation:

\[3x + 5 - 5 = 14 - 5\]
\[3x = 9\]

Next, we can divide both sides of the equation by 3 to solve for \(x\):

\[\frac{3x}{3} = \frac{9}{3}\]
\[x = 3\]

Therefore, the correct answer is **B) 3**.

Reasoning:
- Start with the given equation \(3x + 5 = 14\).
- Subtract 5 from both sides to isolate the term with \(x\): \(3x = 9\).
- Divide both sides by 3 to solve for \(x\), resulting in \(x = 3\).

The other options are incorrect because they either don't satisfy the original equation or lead to incorrect solutions when applied to it. For example, if you choose **A) 2**, you would end up with \(3(2) + 5 = 7\), which does not equal 14 (as required). Similarly, if you choose **C) 4** or **D) 5**, you would get values that aren't consistent with the equation provided.


### Using the SAT Questionbank

We can use a SAT Questionbank (linked here: https://pinesat.com/api/questions) to pick certain questions based on difficulty or domain. This can let us compare how the model performs depending on the type of "thinking" required.

The difficulty levels are "Easy", "Medium", or "Hard".

The available subjects in the question bank are...

    "Information and Ideas" — reading comprehension, main ideas
    "Standard English Conventions" — grammar, punctuation, sentence structure
    "Expression of Ideas" — rhetoric, author's purpose, transitions
    "Craft and Structure" — literary techniques, vocabulary in context

Take a look at the JSON file to get a sense of how the questionbank is structured: https://pinesat.com/api/questions

### Checkpoint #2

In [7]:
# RUN THIS CELL TO ANSWER CHECKPOINT #2
user_response = input("Why do you think we are testing the model with these specific domains? Why not math?\n")
file_path = 'answers.txt'
with open(file_path, 'a') as file:
    file.write(user_response)
    file.write('\n') 

Why do you think we are testing the model with these specific domains? Why not math?
 reading/writing is more subjective or requires more context


First, we will create a function to read the JSON file and fetch a random question for us, depending on the difficulty level and subject area.

In [8]:
import random, requests

def get_sat_question(difficulty = None, subject = None):
    questions = requests.get("https://pinesat.com/api/questions").json()
    if difficulty: questions = [q for q in questions if q["difficulty"].lower() == difficulty.lower()]
    if subject: questions = [q for q in questions if subject.lower() in q["domain"].lower()]
    return random.choice(questions) if questions else None

Here, we didn't specify a difficulty or domain subject. Take this as an example of how to use this function.

In [9]:
q = get_sat_question()
question = q["question"]["question"]
choices = q["question"]["choices"]
passage = q["question"]["paragraph"]
answer = q["question"]["correct_answer"]

# Some questions have a passage associated with them for the testtaker to read before answering the question.
# If that's the case, let's use that as the question instead
if passage and passage != "null":
    question = passage

In [19]:
question

'The text states that "the most common way to reduce carbon emissions is to reduce the use of fossil fuels." The text also states that "the use of fossil fuels is responsible for the vast majority of greenhouse gas emissions." What is the main idea of this passage?'

In [20]:
choices

{'A': 'reducing carbon emissions is difficult due to the prevalence of fossil fuel use.',
 'B': 'reducing carbon emissions is critical to addressing the issue of climate change.',
 'C': 'fossil fuels are the only energy source that produces greenhouse gas emissions.',
 'D': 'fossil fuels are a renewable energy source that needs to be used more responsibly.'}

In [21]:
passage

'The text states that "the most common way to reduce carbon emissions is to reduce the use of fossil fuels." The text also states that "the use of fossil fuels is responsible for the vast majority of greenhouse gas emissions." What is the main idea of this passage?'

In [22]:
question_text = question
options_text = choices

response = ask_sat_question(question_text, options_text)
print(response)

The main idea of this passage is:

Option B: reducing carbon emissions is critical to addressing the issue of climate change.

Reasoning:
- The text mentions that "the most common way to reduce carbon emissions" involves using less fossil fuel. This suggests that reducing fossil fuel usage is crucial in reducing carbon emissions.
- The second statement clarifies that "the use of fossil fuels is responsible for the vast majority of greenhouse gas emissions." This directly addresses the need to reduce carbon emissions.
- There is no mention of reducing carbon emissions being difficult or not possible due to the prevalence of fossil fuel use. Option A is incorrect because it implies that reducing carbon emissions is difficult or impossible.
- Option C is incorrect because it claims that fossil fuels are the only energy source producing greenhouse gases, which is false based on the information provided.
- Option D is incorrect because it suggests that fossil fuels should be used more respo

In [24]:
answer

'B'

### Checkpoint #3

Now, you try! Write some code to pick a random Easy level question and have Qwen attempt the question. Did Qwen get the question right?

In [ ]:
### YOUR CODE HERE

question_text = ...
options_text = ...
response = ask_sat_question(question_text, options_text)
print(response)

In [ ]:
## Did Qwen get the question right? How can you tell? Write the code below.

### YOUR CODE HERE

Let's scale up and create a function to check if Qwen got the right answer automatically. This way, we can run several questions and see how Qwen performs overall.

In [10]:
import re
def extract_answer(response, correct_answer):
    pattern = r"/correct\sanswer\sis\s(.+)\."
    a = re.search(pattern ,answer)
    if correct_answer in response:
        return 'CORRECT'
    else:
        return 'INCORRECT'

In [29]:
extract_answer(response, answer)

'CORRECT'

### 4-Question Performance: Qwen

In [11]:
q1 = get_sat_question('Easy', 'Information and Ideas')
q2 = get_sat_question('Easy', 'Standard English Conventions')
q3 = get_sat_question('Easy', 'Expression of Ideas')
q4 = get_sat_question('Easy', 'Craft and Structure')

q5 = get_sat_question('Medium', 'Information and Ideas')
q6 = get_sat_question('Medium', 'Standard English Conventions')
q7 = get_sat_question('Medium', 'Expression of Ideas')
q8 = get_sat_question('Medium', 'Craft and Structure')

q9 = get_sat_question('Hard', 'Information and Ideas')
q10 = get_sat_question('Hard', 'Standard English Conventions')
q11 = get_sat_question('Hard', 'Expression of Ideas')
q12 = get_sat_question('Hard', 'Craft and Structure')

question_bank = [q1, q2, q3, q4, q5, q6, q7, q8, q9, q10, q11, q12]

In [12]:
import pandas as pd
df = pd.DataFrame(columns = ['Domain', 'Difficulty', 'Question', 'Choices', 'Response', 'Correct?'])
df

,Domain,Difficulty,Question,Choices,Response,Correct?


In [15]:
for q in question_bank:
    question = q["question"]["question"]
    choices = q["question"]["choices"]
    passage = q["question"]["paragraph"]
    answer = q["question"]["correct_answer"]
    domain = q["domain"]
    difficulty = q['difficulty']
    if passage and passage != "null":
        question = passage

    # take the question
    response = ask_sat_question(question, choices)

    #check answer
    check = extract_answer(response, answer)

    #insert it all into a row in the dataframe
    new_row = {'Domain': domain, 'Difficulty': difficulty, 'Question': question, 'Choices': choices, 'Response' : response, 'Correct?': check}
    df.loc[len(df)] = new_row

In [16]:
df

,Domain,Difficulty,Question,Choices,Response,Correct?
0,Information and Ideas,Easy,The artist was renowned for the intricate deta...,"{'A': 'She was meticulous and detail-oriented,...","To determine the correct answer, let's analyze...",CORRECT
1,Standard English Conventions,Easy,A scientist is writing about the process of ph...,"{'A': 'Consequently', 'B': 'However', 'C': 'Mo...",The correct completion of the sentence should ...,CORRECT
2,Expression of Ideas,Easy,One of the most common ways to create a sense ...,{'A': 'create a sense of urgency and excitemen...,"The correct answer is A: ""create a sense of ur...",CORRECT
3,Craft and Structure,Easy,"In the first paragraph, the author describes a...",{'A': 'provide examples of the building’s prac...,The second paragraph serves as a contrast betw...,CORRECT
4,Information and Ideas,Medium,The text focuses on the importance of making c...,{'A': 'create a sense of humor and lighthearte...,The given question is a multiple-choice questi...,CORRECT
5,Standard English Conventions,Medium,The author’s primary goal in this passage is t...,{'A': 'writing is a powerful tool for uncoveri...,The author's primary goal in this passage is t...,CORRECT
6,Expression of Ideas,Medium,The author of the text is attempting to persua...,{'A': 'highlight the author’s concern for the ...,The author of the text is attempting to persua...,CORRECT
7,Craft and Structure,Medium,A large-scale shift in the American economy oc...,{'A': 'explain the causes of the Industrial Re...,The third sentence in the passage provides an ...,INCORRECT
8,Information and Ideas,Hard,The history of the development of the computer...,{'A': 'The abacus was a simple machine that wa...,"Based on the given information, it can be infe...",CORRECT
9,Standard English Conventions,Hard,The director’s approach to the film was innova...,"{'A': 'innovative, using unconventional camera...","The correct answer is:\n\n C: innovative, u...",CORRECT


In [18]:
pd.read_excel('Qwen_results.xlsx')

,Unnamed: 0,Domain,Difficulty,Question,Choices,Response,Correct?
0,0,Information and Ideas,Easy,The author is trying to convey the important m...,"{'A': 'the power of nature.', 'B': 'the import...",The most likely reason why the writer uses the...,CORRECT
1,1,Standard English Conventions,Easy,The main character in the story is a detective...,{'A': 'The passage introduces the detective’s ...,The main idea of the passage is the detective'...,CORRECT
2,2,Expression of Ideas,Easy,The author’s main argument is that the decline...,{'A': 'create a sense of urgency and alarm abo...,"The word ""erosion"" is most likely used in opti...",CORRECT


In [17]:
with open('Qwen_results.xlsx', encoding="utf-8") as f:
    df = pd.read_excel(f)

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc7 in position 15: invalid continuation byte